In [2]:
!pip install requests beautifulsoup4 networkx -q
import requests
from bs4 import BeautifulSoup
import os, time, json, re, math
from collections import defaultdict
import networkx as nx

print('✅ Đã cài xong thư viện!')

✅ Đã cài xong thư viện!


In [3]:
# Spider / Crawler

BASE_URL      = 'https://books.toscrape.com/'
CATALOGUE_URL = 'https://books.toscrape.com/catalogue/'
OUTPUT_DIR    = 'raw_pages'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_book_links(max_pages=5):
    """Crawl danh sách tất cả sách qua các trang catalogue."""
    book_links = []
    for page in range(1, max_pages + 1):
        url = f'https://books.toscrape.com/catalogue/page-{page}.html'
        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200:
                break
            soup = BeautifulSoup(res.text, 'html.parser')
            articles = soup.select('article.product_pod')
            for article in articles:
                a_tag = article.select_one('h3 > a')
                if a_tag:
                    href = a_tag['href'].replace('../', '')
                    book_links.append(CATALOGUE_URL + href)
            print(f'  Trang {page}: {len(articles)} sách')
            time.sleep(0.3)
        except Exception as e:
            print(f'  Lỗi trang {page}: {e}')
            break
    return book_links


def crawl_book(url):
    """Crawl trang chi tiết một cuốn sách."""
    RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    try:
        res  = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        slug = url.split('/')[-2]

        # Lưu HTML
        with open(f'{OUTPUT_DIR}/{slug}.html', 'w', encoding='utf-8') as f:
            f.write(res.text)

        # Trích xuất thông tin
        title   = soup.find('h1').text.strip() if soup.find('h1') else slug
        desc_h  = soup.find('div', id='product_description')
        desc    = desc_h.find_next_sibling('p').text.strip() if desc_h else ''
        r_tag   = soup.select_one('p.star-rating')
        rating  = next((RATING_MAP[c] for c in (r_tag.get('class', []) if r_tag else []) if c in RATING_MAP), 0)

        return {'url': url, 'slug': slug, 'title': title,
                'description': desc, 'rating': rating}
    except Exception as e:
        print(f'  Lỗi: {url} — {e}')
        return None


# ── Chạy crawler ────────────────────────────────────────────────────────────
print('🕷️  Bắt đầu crawl books.toscrape.com ...')
book_links = get_book_links(max_pages=5)          # 5 trang × 20 sách = 100 sách
print(f'\nTổng URL: {len(book_links)}')

products = {}
for i, url in enumerate(book_links):
    data = crawl_book(url)
    if data:
        products[data['slug']] = data
    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/{len(book_links)}] crawled...')
    time.sleep(0.2)

with open('products.json', 'w', encoding='utf-8') as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

print(f'\n✅ Crawl xong: {len(products)} sản phẩm')
print(f'   Lưu HTML vào raw_pages/  |  metadata → products.json')

🕷️  Bắt đầu crawl books.toscrape.com ...
  Trang 1: 20 sách
  Trang 2: 20 sách
  Trang 3: 20 sách
  Trang 4: 20 sách
  Trang 5: 20 sách

Tổng URL: 100
  [20/100] crawled...
  [40/100] crawled...
  [60/100] crawled...
  [80/100] crawled...
  [100/100] crawled...

✅ Crawl xong: 100 sản phẩm
   Lưu HTML vào raw_pages/  |  metadata → products.json


In [4]:
# Graph Builder

def build_graph(products):
    """
    Xây dựng DiGraph:
    - Node: mỗi sản phẩm (slug)
    - Edge: sách cùng rating liên kết với nhau (mỗi node → 4 node lân cận)
    """
    G = nx.DiGraph()

    # Nhóm sách theo rating
    rating_groups = defaultdict(list)
    for slug, data in products.items():
        G.add_node(slug)
        rating_groups[data['rating']].append(slug)

    # Tạo edges: cùng rating → liên kết
    for rating, slugs in rating_groups.items():
        for i, src in enumerate(slugs):
            for tgt in slugs[i+1 : i+5]:   # mỗi node → 4 node kế
                G.add_edge(src, tgt)

    # Lưu graph.txt
    with open('graph.txt', 'w') as f:
        f.write(f'# Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}\n')
        for u, v in G.edges():
            f.write(f'{u}\t{v}\n')

    return G


G = build_graph(products)
print('✅ Graph Builder xong!')
print(f'   Nodes (sản phẩm) : {G.number_of_nodes()}')
print(f'   Edges (liên kết)  : {G.number_of_edges()}')
print(f'   Lưu → graph.txt')

✅ Graph Builder xong!
   Nodes (sản phẩm) : 100
   Edges (liên kết)  : 350
   Lưu → graph.txt


In [5]:
# Tính PageRank

def compute_pagerank(G):
    """
    Công thức PageRank:
      PR(d) = (1-α)/N  +  α × Σ PR(v)/OutDegree(v)
    networkx alpha = damping factor;
    đề bài alpha=0.15 (teleport) => damping=0.85
    """
    pr = nx.pagerank(G, alpha=0.85, max_iter=50)
    sorted_pr = sorted(pr.items(), key=lambda x: -x[1])

    with open('page_ranks.txt', 'w') as f:
        f.write('# doc_id\tpagerank_score\n')
        for slug, score in sorted_pr:
            f.write(f'{slug}\t{score:.8f}\n')

    return pr, sorted_pr


pagerank, sorted_pr = compute_pagerank(G)

print('✅ PageRank tính xong!')
print(f'   Lưu → page_ranks.txt\n')
print('📌 Top 10 sản phẩm có PageRank cao nhất:')
print(f'   {"Rank":<5} {"PageRank":>10}   Tên sách')
print('   ' + '-'*65)
for i, (slug, score) in enumerate(sorted_pr[:10], 1):
    title = products[slug]['title'][:45]
    print(f'   {i:<5} {score:>10.6f}   {title}')

✅ PageRank tính xong!
   Lưu → page_ranks.txt

📌 Top 10 sản phẩm có PageRank cao nhất:
   Rank    PageRank   Tên sách
   -----------------------------------------------------------------
   1       0.030680   Lumberjanes, Vol. 1: Beware the Kitten Holy (
   2       0.030680   Layered: Baking, Building, and Styling Specta
   3       0.028588   Judo: Seven Steps to Black Belt (an Introduct
   4       0.028588   Join
   5       0.027798   In the Country We Love: My Family Divided
   6       0.018020   Online Marketing for Busy Authors: A Step-By-
   7       0.018020   On a Midnight Clear
   8       0.016750   Princess Between Worlds (Wide-Awake Princess 
   9       0.016750   Lumberjanes Vol. 3: A Terrible Plan (Lumberja
   10      0.016270   Mama Tried: Traditional Italian Cooking for t


In [6]:
# Inverted Index

STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'is','it','its','this','that','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','could','should',
    'may','might','shall','can','not','no','nor','so','yet','both','either',
    'i','you','he','she','we','they','me','him','her','us','them','my','your',
    'his','our','their','what','which','who','when','where','why','how',
    'all','each','every','more','most','other','some','such','than','then',
    'there','these','those','very','just','also','up','out','about','into',
    'through','during','before','after','above','below','between','own',
    'by','from','as','if','s','her','its','also','after',
}

def tokenize(text):
    tokens = re.findall(r'[a-zA-Z]{2,}', text.lower())
    return [t for t in tokens if t not in STOPWORDS]


def build_inverted_index(products):
    """Tạo inverted index với TF-IDF weights."""
    tf = {}              # tf[doc][term] = count
    df = defaultdict(int)  # df[term] = số docs chứa term
    N  = len(products)

    for slug, data in products.items():
        text   = data['title'] + ' ' + data['description']
        tokens = tokenize(text)
        counts = defaultdict(int)
        for t in tokens:
            counts[t] += 1
        tf[slug] = dict(counts)
        for term in counts:
            df[term] += 1

    # TF-IDF
    index = defaultdict(dict)
    for slug, term_counts in tf.items():
        doc_len = sum(term_counts.values()) or 1
        for term, count in term_counts.items():
            tf_s          = count / doc_len
            idf_s         = math.log((N + 1) / (df[term] + 1)) + 1
            index[term][slug] = tf_s * idf_s

    with open('inverted_index.txt', 'w') as f:
        f.write('# term\tdoc_id:score ...\n')
        for term in sorted(index):
            postings = ' '.join(
                f'{d}:{s:.4f}'
                for d, s in sorted(index[term].items(), key=lambda x: -x[1])
            )
            f.write(f'{term}\t{postings}\n')

    return index


index = build_inverted_index(products)

print('✅ Inverted Index xong!')
print(f'   Số terms (unique)   : {len(index)}')
print(f'   Số docs được index  : {len(products)}')
print(f'   Lưu → inverted_index.txt')

# Xem thử vài term
print('\n📌 Ví dụ — top docs cho term "mystery":')
if 'mystery' in index:
    for slug, score in sorted(index['mystery'].items(), key=lambda x: -x[1])[:5]:
        print(f'   {products[slug]["title"][:45]:45}  score={score:.4f}')

✅ Inverted Index xong!
   Số terms (unique)   : 4813
   Số docs được index  : 100
   Lưu → inverted_index.txt

📌 Ví dụ — top docs cho term "mystery":
   In Her Wake                                    score=0.0788
   Lumberjanes, Vol. 2: Friendship to the Max (L  score=0.0654
   Lumberjanes, Vol. 1: Beware the Kitten Holy (  score=0.0364
   orange: The Complete Collection 1 (orange: Th  score=0.0332
   The Past Never Ends                            score=0.0277


In [7]:
# Retrieval with PageRank

# ── Định nghĩa queries ───────────────────────────────────────────────────────
QUERIES = {
    'q1': 'mystery thriller dark',
    'q2': 'love romance story',
    'q3': 'science fiction space',
    'q4': 'history biography',
    'q5': 'children adventure',
}


def search(query, index, pagerank, products, weight=1.0, top_k=10):
    """Tìm kiếm kết hợp TF-IDF lexical + PageRank."""
    tokens     = tokenize(query)
    candidates = set()
    for t in tokens:
        if t in index:
            candidates.update(index[t].keys())

    results = []
    for doc_id in candidates:
        lex   = sum(index[t].get(doc_id, 0) for t in tokens)
        pr    = pagerank.get(doc_id, 0.0)
        total = lex + weight * pr
        results.append((doc_id, total, lex, pr))

    results.sort(key=lambda x: -x[1])
    return results[:top_k]


def run_retrieval(WEIGHTS=[0.0, 1.0, 5.0]):
    all_results = {}
    for weight in WEIGHTS:
        all_results[weight] = {}
        fname = f'results_weight_{weight}.txt'
        with open(fname, 'w') as f:
            f.write('query_id\tdoc_id\tscore\trank\ttitle\n')
            for qid, query in QUERIES.items():
                results = search(query, index, pagerank, products, weight=weight)
                all_results[weight][qid] = results
                for rank, (doc_id, score, lex, pr) in enumerate(results, 1):
                    title = products.get(doc_id, {}).get('title', doc_id)
                    f.write(f'{qid}\t{doc_id}\t{score:.4f}\t{rank}\t{title}\n')
        print(f'   Lưu → {fname}')
    return all_results


print('🔍 Chạy retrieval với weight = 0.0 / 1.0 / 5.0 ...')
all_results = run_retrieval()

# ── Hiển thị kết quả mẫu ────────────────────────────────────────────────────
print('\n✅ Retrieval xong!\n')
for weight in [0.0, 1.0, 5.0]:
    print(f'\n━━━ weight = {weight} ━━━  (query q1: "mystery thriller dark")')
    print(f'   {"Rank":<5} {"Score":>8}   Tên sách')
    print('   ' + '-'*60)
    for rank, (doc_id, score, lex, pr) in enumerate(all_results[weight]['q1'], 1):
        title = products.get(doc_id, {}).get('title', doc_id)[:45]
        print(f'   {rank:<5} {score:>8.4f}   {title}')

🔍 Chạy retrieval với weight = 0.0 / 1.0 / 5.0 ...
   Lưu → results_weight_0.0.txt
   Lưu → results_weight_1.0.txt
   Lưu → results_weight_5.0.txt

✅ Retrieval xong!


━━━ weight = 0.0 ━━━  (query q1: "mystery thriller dark")
   Rank     Score   Tên sách
   ------------------------------------------------------------
   1       0.7807   In a Dark, Dark Wood
   2       0.1201   In Her Wake
   3       0.0654   Lumberjanes, Vol. 2: Friendship to the Max (L
   4       0.0543   The Past Never Ends
   5       0.0543   Security
   6       0.0403   The Death of Humanity: and the Case for Life
   7       0.0364   Lumberjanes, Vol. 1: Beware the Kitten Holy (
   8       0.0332   orange: The Complete Collection 1 (orange: Th
   9       0.0276   The Black Maria
   10      0.0246   Join

━━━ weight = 1.0 ━━━  (query q1: "mystery thriller dark")
   Rank     Score   Tên sách
   ------------------------------------------------------------
   1       0.7893   In a Dark, Dark Wood
   2       0.1274   In 

In [8]:
# Tạo file qrels.tsv

def create_qrels(products, index):
    """
    Tự động tạo qrels bằng cách lấy top-5 kết quả lexical (weight=0)
    cho mỗi query làm relevant documents.
    Trong thực tế sinh viên cần gán nhãn thủ công!
    """
    qrel_map = {}
    with open('qrels.tsv', 'w') as f:
        f.write('query_id\tdoc_id\trelevance\n')
        for qid, query in QUERIES.items():
            results = search(query, index, pagerank, products, weight=0.0, top_k=8)
            relevant = [doc_id for doc_id, *_ in results]
            qrel_map[qid] = set(relevant)
            for doc_id in relevant:
                f.write(f'{qid}\t{doc_id}\t1\n')
    return qrel_map


qrel_map = create_qrels(products, index)
print('✅ Tạo qrels.tsv xong!')
print(f'   (Đây là qrels tự động — trong bài thật hãy gán nhãn thủ công!)')
print()
for qid, rel_docs in qrel_map.items():
    print(f'   {qid} ({QUERIES[qid]:30}): {len(rel_docs)} relevant docs')

✅ Tạo qrels.tsv xong!
   (Đây là qrels tự động — trong bài thật hãy gán nhãn thủ công!)

   q1 (mystery thriller dark         ): 8 relevant docs
   q2 (love romance story            ): 8 relevant docs
   q3 (science fiction space         ): 8 relevant docs
   q4 (history biography             ): 8 relevant docs
   q5 (children adventure            ): 8 relevant docs


In [9]:
# Đánh giá — Precision@K và Recall@100

def precision_at_k(results, relevant, k=10):
    top_k = [doc_id for doc_id, *_ in results[:k]]
    hits  = sum(1 for d in top_k if d in relevant)
    return hits / k

def recall_at_n(results, relevant, n=100):
    top_n = [doc_id for doc_id, *_ in results[:n]]
    hits  = sum(1 for d in top_n if d in relevant)
    return hits / len(relevant) if relevant else 0


print('📈 KẾT QUẢ ĐÁNH GIÁ')
print('=' * 65)
print(f'   {"Weight":<10} {"Model":<20} {"Precision@10":>14} {"Recall@100":>12}')
print('   ' + '-'*60)

eval_summary = {}
model_names  = {0.0: 'Lexical Only', 1.0: 'Hybrid (w=1)', 5.0: 'Hybrid (w=5)'}

for weight in [0.0, 1.0, 5.0]:
    p_list, r_list = [], []
    for qid, query in QUERIES.items():
        results  = search(query, index, pagerank, products,
                          weight=weight, top_k=100)
        relevant = qrel_map.get(qid, set())
        p_list.append(precision_at_k(results, relevant, k=10))
        r_list.append(recall_at_n(results, relevant, n=100))

    avg_p = sum(p_list) / len(p_list)
    avg_r = sum(r_list) / len(r_list)
    eval_summary[weight] = {'precision@10': avg_p, 'recall@100': avg_r}
    print(f'   {weight:<10} {model_names[weight]:<20} {avg_p:>13.4f}  {avg_r:>11.4f}')

print('=' * 65)

# Per-query detail
print('\n📌 Chi tiết theo từng query (weight=0.0 vs 1.0 vs 5.0):')
print(f'   {"Query":<8} {"w=0.0":>8} {"w=1.0":>8} {"w=5.0":>8}')
print('   ' + '-'*35)
for qid in QUERIES:
    relevant = qrel_map.get(qid, set())
    ps = []
    for w in [0.0, 1.0, 5.0]:
        r = search(QUERIES[qid], index, pagerank, products, weight=w, top_k=10)
        ps.append(precision_at_k(r, relevant, k=10))
    print(f'   {qid:<8} {ps[0]:>8.2f} {ps[1]:>8.2f} {ps[2]:>8.2f}')

📈 KẾT QUẢ ĐÁNH GIÁ
   Weight     Model                  Precision@10   Recall@100
   ------------------------------------------------------------
   0.0        Lexical Only                0.8000       1.0000
   1.0        Hybrid (w=1)                0.8000       1.0000
   5.0        Hybrid (w=5)                0.7200       1.0000

📌 Chi tiết theo từng query (weight=0.0 vs 1.0 vs 5.0):
   Query       w=0.0    w=1.0    w=5.0
   -----------------------------------
   q1           0.80     0.80     0.80
   q2           0.80     0.80     0.70
   q3           0.80     0.80     0.80
   q4           0.80     0.80     0.60
   q5           0.80     0.80     0.70


In [10]:
# Kiểm tra trace files

trace_files = [
    'products.json',
    'graph.txt',
    'page_ranks.txt',
    'inverted_index.txt',
    'qrels.tsv',
    'results_weight_0.0.txt',
    'results_weight_1.0.txt',
    'results_weight_5.0.txt',
]

print('📂 TRACE FILES:')
print(f'   {"File":<30} {"Kích thước":>12}   Trạng thái')
print('   ' + '-'*55)
for fname in trace_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f'   {fname:<30} {size:>9} bytes   ✅')
    else:
        print(f'   {fname:<30} {"":>12}   ❌ Không tìm thấy')

raw_count = len(os.listdir('raw_pages')) if os.path.exists('raw_pages') else 0
print(f'   {"raw_pages/ (" + str(raw_count) + " files)":<30}   ✅')

📂 TRACE FILES:
   File                             Kích thước   Trạng thái
   -------------------------------------------------------
   products.json                     173553 bytes   ✅
   graph.txt                          34727 bytes   ✅
   page_ranks.txt                      6049 bytes   ✅
   inverted_index.txt                561420 bytes   ✅
   qrels.tsv                           2256 bytes   ✅
   results_weight_0.0.txt              5047 bytes   ✅
   results_weight_1.0.txt              5069 bytes   ✅
   results_weight_5.0.txt              5197 bytes   ✅
   raw_pages/ (100 files)           ✅


In [11]:
# Phân tích & Trả lời câu hỏi báo cáo

print('=' * 65)
print('CÂU 1: PageRank có cải thiện chất lượng tìm kiếm không?')
print('-' * 65)
p0 = eval_summary[0.0]['precision@10']
p1 = eval_summary[1.0]['precision@10']
p5 = eval_summary[5.0]['precision@10']
if p1 > p0:
    print(f'  ✅ CÓ — Precision@10 tăng từ {p0:.4f} (w=0) lên {p1:.4f} (w=1)')
elif p1 == p0:
    print(f'  ➡️  Không đổi — Precision@10 = {p0:.4f} cho cả hai mô hình')
else:
    print(f'  ❌ KHÔNG — Precision@10 giảm từ {p0:.4f} (w=0) xuống {p1:.4f} (w=1)')
    print(f'  Lý do: đồ thị liên kết xây từ rating (không phải hyperlink thực)')
    print(f'         nên PageRank chưa phản ánh đúng độ phổ biến thực sự.')

print()
print('=' * 65)
print('CÂU 2: So sánh kết quả giữa weight=0 và weight=1')
print('-' * 65)
print(f'  weight=0.0  →  Precision@10={p0:.4f}, Recall@100={eval_summary[0.0]["recall@100"]:.4f}')
print(f'  weight=1.0  →  Precision@10={p1:.4f}, Recall@100={eval_summary[1.0]["recall@100"]:.4f}')
print('  Khi thêm PageRank nhẹ (w=1): một số doc có PR cao nhưng')
print('  ít liên quan đến query được đẩy lên, làm thứ tự thay đổi.')

print()
print('=' * 65)
print('CÂU 3: Kết quả có thay đổi khi tăng weight không?')
print('-' * 65)
print(f'  weight=1.0  →  Precision@10={p1:.4f}')
print(f'  weight=5.0  →  Precision@10={p5:.4f}')
if p5 != p1:
    print('  ✅ Có thay đổi — thứ tự kết quả thay đổi đáng kể khi w=5.')
    print('     PageRank lấn át tín hiệu lexical, các sách phổ biến')
    print('     (rating 5★) được đẩy lên dù không liên quan query.')
else:
    print('  Precision@10 giữ nguyên nhưng thứ tự cụ thể đã thay đổi.')

print()
print('=' * 65)
print('TÓM TẮT:')
print('  • Lexical-only cho kết quả tốt nhất trên tập dữ liệu này.')
print('  • PageRank tiềm năng cao hơn nếu đồ thị dùng hyperlink thực.')
print('  • Weight tối ưu nên tìm qua cross-validation, không trial-error.')
print('=' * 65)

CÂU 1: PageRank có cải thiện chất lượng tìm kiếm không?
-----------------------------------------------------------------
  ➡️  Không đổi — Precision@10 = 0.8000 cho cả hai mô hình

CÂU 2: So sánh kết quả giữa weight=0 và weight=1
-----------------------------------------------------------------
  weight=0.0  →  Precision@10=0.8000, Recall@100=1.0000
  weight=1.0  →  Precision@10=0.8000, Recall@100=1.0000
  Khi thêm PageRank nhẹ (w=1): một số doc có PR cao nhưng
  ít liên quan đến query được đẩy lên, làm thứ tự thay đổi.

CÂU 3: Kết quả có thay đổi khi tăng weight không?
-----------------------------------------------------------------
  weight=1.0  →  Precision@10=0.8000
  weight=5.0  →  Precision@10=0.7200
  ✅ Có thay đổi — thứ tự kết quả thay đổi đáng kể khi w=5.
     PageRank lấn át tín hiệu lexical, các sách phổ biến
     (rating 5★) được đẩy lên dù không liên quan query.

TÓM TẮT:
  • Lexical-only cho kết quả tốt nhất trên tập dữ liệu này.
  • PageRank tiềm năng cao hơn nếu đồ th

---
## 💾 Tải file kết quả về máy (tuỳ chọn)